In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# טרנספילציה עם מנהלי מעברים

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
הדרך המומלצת לבצע טרנספילציה של מעגל היא ליצור staged pass manager ולאחר מכן להפעיל את מתודת ה-`run` שלו עם המעגל כקלט. דף זה מסביר כיצד לבצע טרנספילציה של מעגלים קוונטיים בדרך זו.
## מהו (staged) pass manager?
בהקשר של Qiskit SDK, טרנספילציה מתייחסת לתהליך של המרת מעגל קלט לצורה המתאימה להרצה על מכשיר קוונטי. הטרנספילציה מתרחשת בדרך כלל ברצף של שלבים הנקראים transpiler passes. המעגל עובר עיבוד בכל transpiler pass ברצף, כאשר הפלט של pass אחד הופך לקלט של הבא. לדוגמה, pass אחד יכול לעבור על המעגל ולמזג את כל הרצפות הרצופות של שערים חד-Qubit, ואז ה-pass הבא יכול לסנתז שערים אלה לתוך ערכת הבסיס של המכשיר המטרה. transpiler passes הכלולים ב-Qiskit נמצאים במודול [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

pass manager הוא אובייקט שמאחסן רשימה של transpiler passes ויכול להפעיל אותם על מעגל. צור pass manager על ידי אתחול [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) עם רשימה של transpiler passes. כדי להפעיל את הטרנספילציה על מעגל, קרא למתודת [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) עם מעגל כקלט.

staged pass manager הוא סוג מיוחד של pass manager שמייצג רמת הפשטה גבוהה יותר מזו של pass manager רגיל. בעוד שpass manager רגיל מורכב ממספר transpiler passes, staged pass manager מורכב ממספר *pass managers*. זוהי הפשטה שימושית מפני שהטרנספילציה מתרחשת בדרך כלל בשלבים נפרדים, כמתואר ב-[Transpiler stages](transpiler-stages), כאשר כל שלב מיוצג על ידי pass manager. staged pass managers מיוצגים על ידי המחלקה [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). שאר הדף הזה מתאר כיצד ליצור ולהתאים אישית (staged) pass managers.
## יצירת staged pass manager מוגדר מראש
כדי ליצור staged pass manager מוגדר מראש עם ברירות מחדל סבירות, השתמש בפונקציה [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

כדי לבצע טרנספילציה של מעגל או רשימת מעגלים עם pass manager, העבר את המעגל או רשימת המעגלים למתודת `run`. בואו נעשה זאת על מעגל דו-Qubit המורכב מהדמארד ואחריו שני שערי CX סמוכים:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

ראה [Transpilation defaults and configuration options](defaults-and-configuration-options) לתיאור הארגומנטים האפשריים לפונקציה `generate_preset_pass_manager`. הארגומנטים של `generate_preset_pass_manager` תואמים לארגומנטים של פונקציית [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

אם pass managers המוגדרים מראש אינם עונים על צרכיך, התאם אישית את הטרנספילציה על ידי יצירת (staged) pass managers או אפילו transpilation passes. שאר הדף הזה מתאר כיצד ליצור pass managers. להוראות כיצד ליצור transpilation passes, ראה [Write your own transpiler pass](custom-transpiler-pass).

## יצירת pass manager משלך

המודול [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) כולל transpiler passes רבים שניתן להשתמש בהם ליצירת pass managers. כדי ליצור pass manager, אתחל `PassManager` עם רשימה של passes. לדוגמה, הקוד הבא יוצר transpiler pass שממזג שערים דו-Qubit סמוכים ולאחר מכן מסנתז אותם לבסיס של שערי [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) ו-[$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

כדי להדגים את ה-pass manager הזה בפעולה, בדוק אותו על מעגל דו-Qubit המורכב מהדמארד ואחריו שני שערי CX סמוכים:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

כדי להפעיל את ה-pass manager על המעגל, קרא למתודת `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

לדוגמה מתקדמת יותר שמראה כיצד ליצור pass manager לממש את טכניקת דיכוי השגיאות הידועה בשם dynamical decoupling, ראה [Create a pass manager for dynamical decoupling](dynamical-decoupling-pass-manager).

## יצירת staged pass manager

`StagedPassManager` הוא pass manager המורכב משלבים נפרדים, כאשר כל שלב מוגדר על ידי מופע `PassManager`. אפשר ליצור `StagedPassManager` על ידי ציון השלבים הרצויים. לדוגמה, הקוד הבא יוצר staged pass manager עם שני שלבים, `init` ו-`translation`. שלב ה-`translation` מוגדר על ידי ה-pass manager שנוצר קודם.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

אין הגבלה על מספר השלבים שאפשר לשים ב-staged pass manager.

דרך שימושית נוספת ליצירת staged pass manager היא להתחיל עם staged pass manager מוגדר מראש ואז להחליף חלק מהשלבים. לדוגמה, הקוד הבא יוצר pass manager מוגדר מראש עם רמת אופטימיזציה 3, ואז מציין שלב `pre_layout` מותאם אישית.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[פונקציות יוצרות שלבים](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) עשויות להיות שימושיות לבניית pass managers מותאמים אישית.
הן מייצרות שלבים שמספקים פונקציונליות נפוצה שנמצאת בשימוש במנהלי מעברים רבים.
לדוגמה, ניתן להשתמש ב-[`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) ליצירת שלב
ל-"הטמעת" `Layout` ראשוני שנבחר מ-layout pass למכשיר המטרה הנקוב.

## השלבים הבאים
> **Tip:** - [כתוב transpiler pass מותאם אישית](custom-transpiler-pass).
>     - [צור pass manager עבור dynamical decoupling](dynamical-decoupling-pass-manager).
>     - למידע נוסף על הפונקציה `generate_preset_passmanager`, ראה את הנושא [Transpilation default settings and configuration options](defaults-and-configuration-options).
>     - נסה את המדריך [Compare transpiler settings](/guides/circuit-transpilation-settings).
>     - עיין ב-[תיעוד ה-API של ה-Transpiler.](https://docs.quantum.ibm.com/api/qiskit/transpiler)